# VoltRelay Energy — Battery Swap Network Analysis
### Gradient Learnings Data Analytics Hackathon

**Sections:** Setup → Data Understanding → Data Cleaning → EDA → Core Question Analysis (Q1–Q6) → Key Findings & Recommendations

Fill in the `_your interpretation here_` markdown notes after each chart with 2–3 sentences of your own reading of the result — that's what turns this from code into an analysis.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 50)
sns.set_theme(style="whitegrid")

BASE = '/content/drive/MyDrive/voltrelay/'   # <-- update to your actual Drive folder

# ---- auto-save every chart as a separate PNG, no per-cell edits needed ----
import os
os.makedirs('/content/charts', exist_ok=True)
_chart_counter = {'n': 0}
_original_show = plt.show
def _show_and_save(*args, **kwargs):
    _chart_counter['n'] += 1
    fig = plt.gcf()
    fig.savefig(f'/content/charts/chart_{_chart_counter["n"]:02d}.png', dpi=150, bbox_inches='tight')
    _original_show(*args, **kwargs)
plt.show = _show_and_save
# ---------------------------------------------------------------------------

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 150)


## 2. Load Data

In [ ]:
riders    = pd.read_csv(BASE+'riders.csv')
stations  = pd.read_csv(BASE+'stations.csv')
batteries = pd.read_csv(BASE+'batteries.csv')
tickets   = pd.read_csv(BASE+'support_tickets.csv')
hourly    = pd.read_csv(BASE+'station_hourly_status.csv')
city_ctx  = pd.read_csv(BASE+'city_daily_context.csv')
partners  = pd.read_csv(BASE+'fleet_partners.csv')

swap_dtypes = {
    'event_id':'string','rider_id':'string','station_id':'string',
    'event_type':'category','tariff_code':'category','payment_mode':'category',
    'station_firmware':'category','sync_mode':'category',
    'battery_in_id':'string','battery_out_id':'string'
}
swap = pd.read_csv(BASE+'swap_events.csv.gz', dtype=swap_dtypes)

# convert date columns safely (bad values become NaT instead of crashing)
riders['signup_date'] = pd.to_datetime(riders['signup_date'], errors='coerce')

for c in ['commissioned_date','decommissioned_date','firmware_updated_date','competitor_within_1_5km_since']:
    stations[c] = pd.to_datetime(stations[c], errors='coerce')

for c in ['manufacture_date','commission_date','retired_date']:
    batteries[c] = pd.to_datetime(batteries[c], errors='coerce')

tickets['created_ts'] = pd.to_datetime(tickets['created_ts'], errors='coerce')
hourly['hour_start']  = pd.to_datetime(hourly['hour_start'], errors='coerce')
city_ctx['date']      = pd.to_datetime(city_ctx['date'], errors='coerce')

for c in ['contract_start_date','amendment_date']:
    partners[c] = pd.to_datetime(partners[c], errors='coerce')

swap['event_ts'] = pd.to_datetime(swap['event_ts'], errors='coerce')

for df_name, df in [('riders',riders),('stations',stations),('batteries',batteries),
                     ('tickets',tickets),('hourly',hourly),('city_ctx',city_ctx),
                     ('partners',partners),('swap',swap)]:
    print(f'{df_name}: {df.shape}')

## 3. Data Understanding
Quick look at every table before touching anything — size, nulls, and key category distributions.

In [ ]:
for name, df in [('riders',riders),('stations',stations),('batteries',batteries),
                 ('tickets',tickets),('hourly',hourly),('city_ctx',city_ctx),
                 ('partners',partners),('swap',swap)]:
    print(f'\n===== {name} =====')
    print(df.dtypes)
    print('--- % missing ---')
    print((df.isna().mean()*100).round(1))

In [ ]:
print(swap['event_type'].value_counts(normalize=True).round(4)*100)
print()
print(swap['tariff_code'].value_counts())
print()
print(swap['sync_mode'].value_counts())

## 4. Data Cleaning
Each block below fixes one documented data-quality issue. Run them in order — later steps depend on earlier ones.

### 4a. Standardize city spellings (riders.home_city)

In [ ]:
city_map = {
    'Bengaluru':'Bengaluru', 'bengaluru':'Bengaluru', 'BLR':'Bengaluru', 'Bangalore':'Bengaluru',
    'Delhi NCR':'Delhi NCR', 'Delhi':'Delhi NCR', 'New Delhi':'Delhi NCR', 'Gurgaon':'Delhi NCR',
    'Mumbai':'Mumbai', 'Bombay':'Mumbai', 'MUM':'Mumbai',
    'Pune':'Pune', 'pune':'Pune', 'PUN':'Pune',
    'Jaipur':'Jaipur', 'jaipur':'Jaipur', 'JAI':'Jaipur',
    'Hyderabad':'Hyderabad', 'hyderabad':'Hyderabad', 'HYD':'Hyderabad', 'Hyd':'Hyderabad',
}
riders['home_city_clean'] = riders['home_city'].map(city_map)
assert riders['home_city_clean'].isna().sum() == 0, "unmapped city spelling found — check value_counts()"
riders['home_city'].value_counts()

### 4b. Fix the firmware timestamp bug
Stations on firmware `v3.2.0` logged events ~5h30m early between 2025-03-10 and 2025-04-14.

In [ ]:
bug_mask = (swap['station_firmware']=='v3.2.0') & \
           (swap['event_ts'].between('2025-03-10','2025-04-14 23:59:59'))
print('rows affected:', bug_mask.sum())
swap.loc[bug_mask, 'event_ts'] = swap.loc[bug_mask, 'event_ts'] + pd.Timedelta(hours=5, minutes=30)

### 4c. Remove offline-sync duplicate events
Near-duplicate rows: same rider + station + battery, `sync_mode='offline_batch'`, timestamps a few seconds/minutes apart.

In [ ]:
swap = swap.sort_values('event_ts').reset_index(drop=True)
dup_key = ['rider_id','station_id','battery_in_id']
swap['prev_ts'] = swap.groupby(dup_key, observed=True)['event_ts'].shift(1)
swap['gap_sec'] = (swap['event_ts'] - swap['prev_ts']).dt.total_seconds()

is_dup = (swap['sync_mode']=='offline_batch') & (swap['gap_sec'] <= 150)
print('duplicate rows removed:', is_dup.sum())

swap = swap[~is_dup].drop(columns=['prev_ts','gap_sec']).reset_index(drop=True)

### 4d. Flag (don't silently drop) sensor/odometer outliers

In [ ]:
swap['km_flag'] = (swap['km_since_last_swap'] < 0) | (swap['km_since_last_swap'] > 500)
for c in ['soc_in_pct','soc_out_pct','soh_in_pct','soh_out_pct']:
    swap[c+'_flag'] = swap[c] > 100

print('km outliers:', swap['km_flag'].sum())
print('soc/soh >100% readings:',
      {c: int(swap[c+'_flag'].sum()) for c in ['soc_in_pct','soc_out_pct','soh_in_pct','soh_out_pct']})

### 4e. Exclude internal test stations from network-performance analysis

In [ ]:
stations_real = stations[~stations['station_id'].str.startswith('STN-TST')].copy()
print('test stations excluded:', stations['station_id'].str.startswith('STN-TST').sum())

### 4f. Split completed vs. all attempts
Most pricing / battery-condition fields are only populated for `swap_completed` — keep both views.

In [ ]:
completed = swap[swap['event_type']=='swap_completed'].copy()
print('all attempts:', len(swap), '| completed:', len(completed))

## 5. Build the analysis-ready merged table

In [ ]:
swap_m = (swap
    .merge(riders[['rider_id','partner_id','vehicle_class','home_city_clean','signup_date','plan_type']],
           on='rider_id', how='left')
    .merge(stations_real, on='station_id', how='inner', suffixes=('','_station'))  # inner drops test stations
)
swap_m['event_date'] = swap_m['event_ts'].dt.date
swap_m['event_month'] = swap_m['event_ts'].dt.to_period('M').astype(str)
swap_m['event_hour']  = swap_m['event_ts'].dt.hour

def season_of(month):
    m = int(month.split('-')[1])
    if m in [12,1,2]: return 'winter'
    if m in [3,4,5]: return 'summer'
    if m in [6,7,8,9]: return 'monsoon'
    return 'post_monsoon'
swap_m['season'] = swap_m['event_month'].apply(season_of)

completed_m = swap_m[swap_m['event_type']=='swap_completed'].copy()
print(swap_m.shape)

## 6. Exploratory Data Analysis

In [ ]:
monthly = swap_m.groupby('event_month').agg(
    attempts=('event_id','count'),
    completed=('event_type', lambda s: (s=='swap_completed').sum())
).reset_index()
monthly['failure_rate'] = 1 - monthly['completed']/monthly['attempts']

fig, ax1 = plt.subplots(figsize=(11,4))
ax1.plot(monthly['event_month'], monthly['attempts'], marker='o', label='Attempts')
ax1.set_ylabel('Swap attempts')
ax1.tick_params(axis='x', rotation=90)
ax2 = ax1.twinx()
ax2.plot(monthly['event_month'], monthly['failure_rate']*100, color='red', marker='o', label='Failure rate %')
ax2.set_ylabel('Failure rate (%)')
fig.suptitle('Monthly swap volume vs. failure rate')
fig.legend(loc='upper left')
plt.tight_layout()
plt.show()

_Your interpretation here: does volume growth track with failure rate, or diverge?_

## 7. Core Question 1 — Network Performance Over Time
Completed swaps, revenue, failure rate, and contribution margin per swap, trended monthly.

In [ ]:
q1 = swap_m.groupby('event_month').apply(lambda g: pd.Series({
    'completed_swaps': (g['event_type']=='swap_completed').sum(),
    'revenue_inr': g.loc[g['event_type']=='swap_completed','amount_charged_inr'].sum(),
    'failure_rate': 1 - (g['event_type']=='swap_completed').mean(),
    'avg_energy_cost_inr': (g.loc[g['event_type']=='swap_completed','energy_to_recharge_kwh']
                             * g.loc[g['event_type']=='swap_completed','grid_tariff_inr_kwh']).mean()
})).reset_index()

q1['revenue_per_swap'] = q1['revenue_inr'] / q1['completed_swaps']
q1['margin_per_swap'] = q1['revenue_per_swap'] - q1['avg_energy_cost_inr']

fig, axes = plt.subplots(2,2, figsize=(13,7))
axes[0,0].plot(q1['event_month'], q1['completed_swaps'], marker='o'); axes[0,0].set_title('Completed swaps/month'); axes[0,0].tick_params(axis='x', rotation=90)
axes[0,1].plot(q1['event_month'], q1['revenue_inr'], marker='o', color='green'); axes[0,1].set_title('Revenue/month'); axes[0,1].tick_params(axis='x', rotation=90)
axes[1,0].plot(q1['event_month'], q1['failure_rate']*100, marker='o', color='red'); axes[1,0].set_title('Failure rate %'); axes[1,0].tick_params(axis='x', rotation=90)
axes[1,1].plot(q1['event_month'], q1['margin_per_swap'], marker='o', color='purple'); axes[1,1].set_title('Contribution margin / swap (INR)'); axes[1,1].tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.show()

q1

_Your interpretation here: do all four metrics move together, or does margin/failure diverge from volume/revenue?_

## 8. Core Question 2 — Service Failures & Customer Experience
Queue wait, failed attempts, abandonment vs. station / hour / season / vehicle class.

In [ ]:
station_perf = swap_m.groupby('station_id').agg(
    attempts=('event_id','count'),
    failure_rate=('event_type', lambda s: 1-(s=='swap_completed').mean()),
    avg_queue_wait=('queue_wait_sec','mean')
).reset_index().sort_values('failure_rate', ascending=False)

plt.figure(figsize=(9,4))
sns.histplot(station_perf['failure_rate']*100, bins=30)
plt.title('Distribution of failure rate across stations')
plt.xlabel('Failure rate (%)')
plt.show()

station_perf.head(15)

In [ ]:
hour_perf = swap_m.groupby('event_hour').agg(
    failure_rate=('event_type', lambda s: 1-(s=='swap_completed').mean()),
    avg_queue_wait=('queue_wait_sec','mean')
).reset_index()

fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.barplot(data=hour_perf, x='event_hour', y='failure_rate', ax=ax[0]); ax[0].set_title('Failure rate by hour of day')
sns.barplot(data=hour_perf, x='event_hour', y='avg_queue_wait', ax=ax[1]); ax[1].set_title('Avg queue wait by hour of day')
plt.tight_layout(); plt.show()

In [ ]:
seg = swap_m.groupby(['season','vehicle_class']).agg(
    failure_rate=('event_type', lambda s: 1-(s=='swap_completed').mean()),
    avg_queue_wait=('queue_wait_sec','mean')
).reset_index()
seg

_Your interpretation here: is service quality spread evenly, or concentrated at specific stations/hours/segments?_

## 9. Core Question 3 — Station & Geographic Patterns
Charger generation, location type, and commissioning timing vs. performance.

In [ ]:
geo = swap_m.groupby(['charger_generation','location_type']).agg(
    attempts=('event_id','count'),
    failure_rate=('event_type', lambda s: 1-(s=='swap_completed').mean()),
    avg_queue_wait=('queue_wait_sec','mean')
).reset_index().sort_values('failure_rate', ascending=False)
geo

In [ ]:
# turnaround time from station_hourly_status, joined by station
hourly_s = hourly.merge(stations_real[['station_id','charger_generation','location_type','city']], on='station_id', how='inner')
turnaround = hourly_s.groupby('charger_generation')['avg_charge_minutes'].mean().reset_index()

plt.figure(figsize=(7,4))
sns.barplot(data=turnaround, x='charger_generation', y='avg_charge_minutes')
plt.title('Avg charge turnaround time by charger generation')
plt.show()

In [ ]:
# complaints per station, normalized by attempt volume
complaints = tickets.groupby('station_id').size().rename('ticket_count').reset_index()
complaint_rate = station_perf.merge(complaints, on='station_id', how='left').fillna({'ticket_count':0})
complaint_rate['tickets_per_1000_attempts'] = complaint_rate['ticket_count']/complaint_rate['attempts']*1000
complaint_rate.sort_values('tickets_per_1000_attempts', ascending=False).head(15)

_Your interpretation here: which equipment/location combinations stand out, good or bad?_

## 10. Core Question 4 — Battery & Equipment Performance
State of health, charge cycles, supplier, and manufacturing lot vs. delivered range and swap frequency.

In [ ]:
bat_usage = completed_m.groupby('battery_out_id').agg(
    times_issued=('event_id','count'),
    avg_soc_out=('soc_out_pct','mean')
).reset_index().rename(columns={'battery_out_id':'battery_id'})

bat_full = batteries.merge(bat_usage, on='battery_id', how='left')

supplier_perf = bat_full.groupby('supplier').agg(
    avg_current_soh=('current_soh_pct','mean'),
    avg_times_issued=('times_issued','mean'),
    avg_soc_out=('avg_soc_out','mean')
).reset_index()

plt.figure(figsize=(8,4))
sns.barplot(data=supplier_perf, x='supplier', y='avg_current_soh')
plt.title('Average current SoH by supplier')
plt.show()

supplier_perf

In [ ]:
lot_perf = bat_full.groupby('manufacturing_lot').agg(
    avg_current_soh=('current_soh_pct','mean'),
    count=('battery_id','count')
).reset_index().sort_values('avg_current_soh')
lot_perf[lot_perf['count']>=10].head(15)   # ignore tiny lots for now

_Your interpretation here: does any supplier or manufacturing lot clearly underperform?_

## 11. Core Question 5 — Pricing & Partner Economics
Base price change, peak/off-peak pilot, and fleet contract terms vs. timing, revenue, margin.

In [ ]:
tariff_perf = completed_m.groupby('tariff_code').agg(
    swaps=('event_id','count'),
    avg_amount=('amount_charged_inr','mean'),
    avg_discount=('discount_inr','mean')
).reset_index()
tariff_perf

In [ ]:
partner_riders = riders[riders['partner_id'].notna()][['rider_id','partner_id']]
partner_swaps = completed_m.merge(partner_riders, on='rider_id', how='inner') \
                            .merge(partners, on='partner_id', how='left')

partner_econ = partner_swaps.groupby('partner_name').agg(
    swaps=('event_id','count'),
    revenue=('amount_charged_inr','sum'),
    avg_energy_cost=('energy_to_recharge_kwh','mean')
).reset_index().sort_values('revenue', ascending=False)
partner_econ

### Before/after the price change or contract amendment
Pick the actual base-price-change date and each partner's `amendment_date` from your data, then compare margin on either side.

In [ ]:
# example pattern — replace CHANGE_DATE with the real one you find in city_ctx/pricing notes
CHANGE_DATE = '2024-09-01'  # <-- update after checking when tariff/list_price shifted
before = completed_m[completed_m['event_ts'] < CHANGE_DATE]['amount_charged_inr'].mean()
after  = completed_m[completed_m['event_ts'] >= CHANGE_DATE]['amount_charged_inr'].mean()
print('avg amount before:', round(before,2), '| after:', round(after,2))

_Your interpretation here: which pricing/contract levers actually moved revenue or margin, and for whom?_

## 12. Core Question 6 — Root Cause of Retention/Churn
Define a "returned" rider, then compare candidate churn factors side by side.

In [ ]:
first_swap = swap_m.groupby('rider_id')['event_ts'].min().rename('first_swap_ts')
last_swap  = swap_m.groupby('rider_id')['event_ts'].max().rename('last_swap_ts')
swap_count = swap_m.groupby('rider_id')['event_id'].count().rename('total_attempts')

rider_activity = riders.merge(first_swap, on='rider_id', how='left') \
                        .merge(last_swap, on='rider_id', how='left') \
                        .merge(swap_count, on='rider_id', how='left')

# "new rider" = signed up in the data window; "returned" = swapped again 30+ days after first swap
rider_activity['days_active'] = (rider_activity['last_swap_ts'] - rider_activity['first_swap_ts']).dt.days
rider_activity['returned_30d'] = rider_activity['days_active'] >= 30
rider_activity['returned_30d'] = rider_activity['returned_30d'].fillna(False)

print(rider_activity['returned_30d'].mean())

In [ ]:
# candidate factor 1: did their first swap fail?
first_event_type = swap_m.sort_values('event_ts').groupby('rider_id')['event_type'].first().rename('first_event_type')
rider_activity = rider_activity.merge(first_event_type, on='rider_id', how='left')
rider_activity['first_swap_failed'] = rider_activity['first_event_type'] != 'swap_completed'

factor1 = rider_activity.groupby('first_swap_failed')['returned_30d'].mean()
print('Return rate by whether first attempt failed:\n', factor1)

In [ ]:
# candidate factor 2: station quality at signup (avg failure rate of stations they used)
rider_station_quality = swap_m.merge(station_perf[['station_id','failure_rate']], on='station_id', how='left') \
                               .groupby('rider_id')['failure_rate'].mean().rename('avg_station_failure_rate')
rider_activity = rider_activity.merge(rider_station_quality, on='rider_id', how='left')

# candidate factor 3: did they raise a support ticket early on?
early_tickets = tickets.merge(rider_activity[['rider_id','first_swap_ts']], on='rider_id', how='inner')
early_tickets = early_tickets[early_tickets['created_ts'] <= early_tickets['first_swap_ts'] + pd.Timedelta(days=14)]
had_early_ticket = early_tickets.groupby('rider_id').size().rename('early_ticket_count')
rider_activity = rider_activity.merge(had_early_ticket, on='rider_id', how='left')
rider_activity['had_early_ticket'] = rider_activity['early_ticket_count'].fillna(0) > 0

factor3 = rider_activity.groupby('had_early_ticket')['returned_30d'].mean()
print('Return rate by early support ticket:\n', factor3)

In [ ]:
# compare effect sizes of all candidate factors side by side
factors_summary = pd.DataFrame({
    'first_swap_failed': rider_activity.groupby('first_swap_failed')['returned_30d'].mean(),
}).T
print(factors_summary)
# add more candidate factors here: plan_type, signup_channel, vehicle_class, tariff exposure, etc.
for f in ['plan_type','signup_channel','vehicle_class']:
    print(f'\n--- return rate by {f} ---')
    print(rider_activity.groupby(f)['returned_30d'].mean().sort_values(ascending=False))

_Your interpretation here: rank the factors by how strongly they separate returners from non-returners, and call out which look primary vs. secondary._

## 13. Key Findings & Recommendations

**Network performance:** _fill in from Q1_

**Service quality:** _fill in from Q2/Q3_

**Equipment:** _fill in from Q4_

**Pricing & partners:** _fill in from Q5_

**Retention root cause:** _fill in from Q6, ranked by strength of evidence_

**Recommendations for leadership (tie each to the budget-decision options — more stations, more batteries, pricing rollout, fleet exclusive):**
1. _..._
2. _..._
3. _..._


## 14. Download all charts as one ZIP
Run this last, after every chart cell above has been executed at least once.


In [ ]:
import shutil
from google.colab import files

print(f"Charts saved so far: {_chart_counter['n']}")
shutil.make_archive('/content/all_charts', 'zip', '/content/charts')
files.download('/content/all_charts.zip')
